In [1]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess

In [2]:
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")

Groq API Key exists and begins gsk_uqLa


In [3]:
groq_client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
models = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct-0905",
    "openai/gpt-oss-120b",
    "qwen/qwen3-32b"
]

clients = {model: groq_client for model in models}

In [5]:
from system_info import retrieve_system_info
system_info = retrieve_system_info()
print(system_info)

{'os': {'system': 'Linux', 'arch': 'x86_64', 'release': '6.6.87.2-microsoft-standard-WSL2', 'version': '#1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025', 'kernel': '6.6.87.2-microsoft-standard-WSL2', 'distro': {'name': 'Ubuntu 24.04.3 LTS', 'version': '24.04'}, 'wsl': True, 'rosetta2_translated': False, 'target_triple': 'x86_64-linux-gnu'}, 'package_managers': ['apt'], 'cpu': {'brand': 'Intel(R) Core(TM) i5-6200U CPU @ 2.30GHz', 'cores_logical': 4, 'cores_physical': 2, 'simd': ['AVX', 'AVX2', 'FMA', 'SSE4_2']}, 'toolchain': {'compilers': {'gcc': 'gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0', 'g++': 'g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0', 'clang': '', 'msvc_cl': ''}, 'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 4.3'}, 'linkers': {'ld_lld': ''}}}


In [6]:
compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "-flto", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [7]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [8]:
def port(model, python):
    client = clients[model]
    kwargs = {
        "model": model,
        "messages": messages_for(python)
    }
    
    if model == "openai/gpt-oss-120b":
        kwargs["reasoning_effort"] = "medium"
    elif model == "qwen/qwen3-32b":
        kwargs["reasoning_effort"] = None
    
    response = client.chat.completions.create(**kwargs)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    write_output(reply)
    return reply

In [9]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [10]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout
    return output

In [11]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [12]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


gio: http://127.0.0.1:7860/: Operation not supported


In [19]:
print(compile_and_run())

An error occurred:
main.cpp:2:85: warning: missing terminating ' character
    2 | Okay, I need to convert this Python code to C++ and make it as fast as possible. Let's look at the Python code first.
      |                                                                                     ^
main.cpp:2:85: error: missing terminating ' character
    2 | Okay, I need to convert this Python code to C++ and make it as fast as possible. Let's look at the Python code first.
      |                                                                                     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
main.cpp:4:98: error: too many decimal points in number
    4 | The main function is calculate, which takes iterations, param1, param2. It initializes result to 1.0. Then, for each i from 1 to iterations, it computes j as i*param1 minus param2, subtracts 1/j from result. Then computes j as i*param1 plus param2, adds 1/j to result. Finally, returns result. The main part runs calculate with 200 mil

llama 3.1-8b - error <br>
llama 3.3-70b - error <br>
llama 4- error<br>
kimi-k2 - error<br>
gpt-oss - 0.736257<br>
qwen error<br>